# CASSETTE
- segment MIDI both on bar & on beat level: `pretty_midi`
- extract weighted pitch class profile for each of the segments of MIDI file
    - extract notes sounding between start & end time
    - compute, for each note, product of MIDI velocity & proportion of segment during which note sounds
    - sums product over all notes in same pitch class
    - normalize weighted pitch class profile by dividing each element by total sum of all its elements (makes feature invariant to total loudness & duration of notes in segment)
- find best matching chord for each segment, by assigning chord template that's most similar to normalized weighted pitch class profile of segment
    - vocabulary of 25 chords (24 maj/min chords + no-chord symbol)
    - chord template is 12-dim vector
    - similar score using Pardo & Birmingham $S = P - (N + M)$ where $P$ = sum of weights of pitch classes of bar that match a template element, $N$ = sum of weights of pitch classes of bar that do not match a template element, $M$ = count of template elements not matched by any note
    - chord with highest template similarity score is assigned
    - if score <= -3, algorithm assigned no-chord
    - if multiple templates have same similarity score, selects template whose root pitch has greatest weight in segment's pitch class profile

##Install and import packages, set up Drive for Dataset

In [1]:
# run in base_environment
!pip install pretty_midi mir_eval mirdata pyfluidsynth

# Packages
import pretty_midi
import mirdata
import mir_eval
import numpy as np
import pandas as pd
from pathlib import Path

# Our functions
import utils as u

# For plotting
import mir_eval.display
import librosa.display
import matplotlib.pyplot as plt

# For audio display
from IPython.display import Audio

## Initialize loader, download and load data using mirdata





For this project we will use SLAKH datset, a synthesized version of the LAKH dataset:
<blockquote>
Manilow, Ethan, Gordon Wichern, Prem Seetharaman, and Jonathan Le Roux. "Cutting music source separation some Slakh: A dataset to study the impact of training data quality and quantity." In 2019 IEEE Workshop on Applications of Signal Processing to Audio and Acoustics (WASPAA), pp. 45-49. IEEE, 2019.
</blockquote>

In [2]:
# Mount Drive to work with dataset
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [3]:
# Location of the datset (using small subset for now)
data_home = '/content/drive/MyDrive/slakh_demo'
dataset_name = 'slakh'
dataset_version = 'baby'
dataset = u.load_data(dataset_name, data_home=data_home, dataset_version=dataset_version)

# Run the following line once to download
#dataset.download()

# Download the index for mirdata to load
#dataset.download(partial_download=['index'])

# Uncomment the following line and run to validate
#dataset.validate()


### Sample a random multitrack and generate audio:

In [4]:
# Sample a track and check Audio
example = dataset.choice_multitrack()

# listen to Dataset audio
Audio(example.audio[0], rate=example.audio[1])
# Listen to Synthesized Audio
#synth = example.midi.synthesize(fs=44100)
#Audio(synth, rate=44100)

Output hidden; open in https://colab.research.google.com to view.

## Step 1a: Segment the MIDI files by beats and downbeats:


In [5]:
# Load all multitracks
data = dataset.load_multitracks()

# Get downbeats and beats for dataset using PrettyMIDI function
downbeats = u.beat_times(data, division='downbeat')
beats = u.beat_times(data, division='beat')

/usr/local/lib/python3.12/dist-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


## Step 1b: Using beat and downbeat times, find segments of MIDI and store active notes

segment_midi() takes the dataset dictionary and the dictionary of beat times, storing an collection of values ('start', 'end', 'notes') for each segment of each track. All three dictionaries are keyed by mtrack_id

In [6]:
# Get a dictionary of segment information for the dataset
beat_segments = u.segment_midi(data, beats)
dbeat_segments = u.segment_midi(data, downbeats)


###Create click track to check alignment of audio to midi segments?
 I haven't modified this since Estee pushed, I think I broke it

In [7]:
def create_click_waveform(freq, fs=44100, duration=0.05):
    t = np.linspace(0, duration, int(fs * duration))
    # generate sine wave
    click = np.sin(2 * np.pi * freq * t)
    # apply exponential decay (the envelope) so it sounds like a 'click'
    envelope = np.exp(-t / (0.01 * duration))
    return click * envelope

regular_waveform = create_click_waveform(440)

In [16]:
audio = dataset.choice_multitrack().audio

beat_clicks = mir_eval.sonify.clicks(beats, fs=audio[1], length=len(audio[0]), click=regular_waveform)
downbeat_clicks = mir_eval.sonify.clicks(downbeats, fs=audio[1], length=len(audio[0]))
Audio(audio[0] + beat_clicks + downbeat_clicks, rate=audio[1])

ValueError: invalid literal for int() with base 10: 'Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track00001Track0000

## Step 2a: Calculate the weighted Pitch Class Profile scores for each segment of each multitrack, relative to the velocity and duration of the notes active during the segment:

In [9]:
beat_pcp = u.weighted_pitch_class(beat_segments)
dbeat_pcp = u.weighted_pitch_class(dbeat_segments)

## Step 2b: Define chord templates and compare Pitch Class Profile scores for each segment against all templates, finding the best chord estimate and similarity score

The CASSETTE paper used only major and minor triad templates, along with a No Chord template. At the moment, we have the same, but could be scaled up to a larger vocabulary in the following cell:

In [10]:
# define chord templates
# start with basic maj/min + 'N' chords
CHORD_PATTERNS = {
    'maj': [1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0],
    'min': [1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0]
}

templates = u.get_all_templates(CHORD_PATTERNS)

<i>classify_chord()</i> from Utils.py uses function <i>calc_similarity()</i> to compare similarity scores for each template and store the best chord estimate + corresponding similarity score for each segment of a multitrack.

In [11]:
# Create dictionaries to store estimates for all multitracks
# Each key will have chord array and similarity score array (n tracks x (2 x m segments))
beat_estimates = {}
dbeat_estimates = {}

for id, pcp in beat_pcp.items():
    beat_estimates[id] = u.classify_chord(pcp, templates)

for id, pcp in dbeat_pcp.items():
    dbeat_estimates[id] = u.classify_chord(pcp, templates)


In [15]:
# Test that estimates are working
db_chordlen = len(dbeat_estimates['Track00016'][0])
db_simlen = len(dbeat_estimates['Track00016'][1])
b_chordlen = len(beat_estimates['Track00016'][0])
b_simlen = len(beat_estimates['Track00016'][1])

if  db_chordlen == db_simlen:
  print(f"Downbeat chords and similarities same length: {db_chordlen}")
if  b_chordlen == b_simlen:
  print(f"Beat chords and similarities same length: {b_chordlen}")

print(dbeat_estimates['Track00016'][0])
print(beat_estimates['Track00016'][0])

Downbeat chords and similarities same length: 84
Beat chords and similarities same length: 339
['N', 'N', 'D#:maj', 'D#:maj', 'F:min', 'F:min', 'C:min', 'F:maj', 'C:min', 'F:maj', 'C:min', 'F:maj', 'C:min', 'F:maj', 'C:min', 'G#:maj', 'G#:maj', 'G:maj', 'C:min', 'F:maj', 'C:min', 'F:maj', 'C:min', 'G#:maj', 'G#:maj', 'A#:maj', 'A#:maj', 'D#:maj', 'A#:maj', 'D#:maj', 'G:min', 'D#:maj', 'D#:maj', 'F:min', 'C#:maj', 'G#:min', 'D#:maj', 'G#:min', 'D#:maj', 'C:min', 'F:maj', 'C:min', 'F:maj', 'C:min', 'G#:maj', 'G#:maj', 'A#:maj', 'A#:maj', 'D#:maj', 'A#:maj', 'D#:maj', 'G:min', 'D#:maj', 'D#:maj', 'F:min', 'C#:maj', 'G#:min', 'D#:maj', 'F#:maj', 'C#:maj', 'G#:min', 'D#:min', 'F#:maj', 'C#:maj', 'G#:min', 'D#:maj', 'A#:maj', 'C:maj', 'C:min', 'C:maj', 'D#:maj', 'D#:maj', 'C:min', 'F:min', 'G#:min', 'D#:maj', 'D#:maj', 'F:min', 'C#:maj', 'G#:min', 'D#:maj', 'G#:min', 'D#:maj', '']
['N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'N', 'D#:maj', 'D#:maj', 'D#:maj', 'D#:maj', 'D#:maj', 'D#:maj', '

# Part C: Results and evaluation

FYI: chord estimates and respective similarity scores are stored in dictionaries <i>dbeat_estimates</i> and <i>beat_estimates</i> at indices [0] and [1], respectively.

### evaluate against chord recognizer

In [ ]:
# get saved crema output
crema_output = pd.read_csv('sample_output.csv')
crema_output.head()

In [ ]:
crema_output['end_time'] = crema_output['time'] + crema_output['duration']
crema_output = crema_output.rename(columns={'time':'start_time'})
crema_output.head()

In [ ]:
print(len(beats))
print(len(results))

In [ ]:
beats_df = pd.DataFrame({'end_time':beats})
beats_df['start_time'] = beats_df['end_time'].shift(1)
beats_df = beats_df[['start_time', 'end_time']].iloc[1:, :]
beats_df.head()

In [ ]:
chord_df = pd.DataFrame({'chord':[i[0] for i in results]})
chord_df.head()

In [ ]:
df = pd.concat([beats_df, chord_df], axis=1)
df.head(15)

In [ ]:
crema_output.head()

In [ ]:
crema_output['value'].unique()

In [ ]:
def normalize_chord_label(label):
    if label == 'N':
        return 'N'

    root, quality = label.split(':')
    if 'min' in quality or 'hdim' in quality or 'dim' in quality or '7' in quality:
        quality = 'min'
    elif 'maj' in quality:
        quality = 'maj'

    return f'{root}:{quality}'

In [ ]:
[normalize_chord_label(i) for i in crema_output['value'].unique()]